# 11 — Human-in-the-Loop Approval

Pause a LangGraph mid-run at an `interrupt()` call, inspect the proposed action, and resume with `Command(resume=True/False)` — all from a notebook cell.

**What you'll learn**
- `interrupt(value)` — suspends the graph and surfaces a value to the caller (like `yield` for graphs)
- `MemorySaver` — required checkpointer that persists state across the two `.stream()` calls
- `thread_id` — ties both calls to the same saved checkpoint so resume finds the right state
- `Command(resume=decision)` — second `.stream()` call that picks up exactly where the graph stopped
- Why `await_approval` re-runs from the top on resume (and what that means for side effects)

**Companion examples:** 5-react-agent-lg (MemorySaver for multi-turn memory), 30-agentic-rag (agent + tools)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Build the HITL graph ────────────────────────────────────────────────────
# Two nodes: draft_action prepares the action, await_approval is the pause gate.
#
# interrupt() is the key primitive:
#   - Pauses the graph mid-node and surfaces a value to the caller
#   - State is saved to the checkpointer
#   - On resume, the node re-runs from the TOP — put any side effects AFTER interrupt()
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import interrupt

PENDING_ACTION = "Delete all records older than 90 days from the analytics database"


class HITLState(TypedDict):
    action: str
    approved: bool
    result: str


def draft_action(state: HITLState) -> dict:
    return {"action": PENDING_ACTION}


def await_approval(state: HITLState) -> dict:
    # Everything BEFORE interrupt() runs twice (initial + resume).
    # Only put side effects AFTER the interrupt() call.
    decision = interrupt({"action": state["action"], "question": "Approve this action?"})
    if decision:
        return {"approved": True, "result": f"EXECUTED: {state['action']}"}
    return {"approved": False, "result": f"CANCELLED: {state['action']}"}


graph = StateGraph(HITLState)
graph.add_node("draft_action", draft_action)
graph.add_node("await_approval", await_approval)
graph.add_edge(START, "draft_action")
graph.add_edge("draft_action", "await_approval")
graph.add_edge("await_approval", END)

# MemorySaver is REQUIRED — without it the graph cannot resume from the interrupt
app = graph.compile(checkpointer=MemorySaver())

print("Graph compiled — MemorySaver checkpointer active")

In [ ]:
# ── 4. First .stream() — runs until interrupt() fires ─────────────────────────
# Unlike .invoke() which blocks until END, this yields a __interrupt__ event and stops.
# The graph state is saved; we can resume later.
config = {"configurable": {"thread_id": "hitl-demo"}}

interrupted = None
for chunk in app.stream(
    {"action": "", "approved": False, "result": ""},
    config,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        interrupted = chunk["__interrupt__"][0].value
        break
    for node, update in chunk.items():
        if update.get("action"):
            print(f"[{node}] Drafted: {update['action']}")

if interrupted:
    print(f"\nPAUSED — awaiting human decision")
    print(f"  Action  : {interrupted['action']}")
    print(f"  Prompt  : {interrupted['question']}")

In [ ]:
# ── 5. Make a decision, then resume ───────────────────────────────────────────
# Change APPROVED = False to see the cancellation path.
# In a real app this would be a UI button or webhook.
from langgraph.types import Command

APPROVED = True  # ← change to False and re-run to see the cancel path

print(f"Resuming with: {'APPROVED' if APPROVED else 'REJECTED'}\n")

# Command(resume=...) tells the checkpointer to replay saved state,
# re-run await_approval, and have interrupt() return APPROVED this time.
for chunk in app.stream(Command(resume=APPROVED), config, stream_mode="updates"):
    for node, update in chunk.items():
        if update.get("result"):
            print(f"[{node}] {update['result']}")

## Exercises

**Exercise 1 — Reject the action:** Change `APPROVED = False` in cell 5. Re-run cells 4 and 5 with a fresh `thread_id` (e.g., `"hitl-demo-2"`). What result string does the graph produce?

**Exercise 2 — Inspect checkpoint state:** After running cell 4 (before cell 5), call `app.get_state(config)`. You should see the suspended state — `action` filled in, `approved=False`, `result=""`. This is what MemorySaver saved.

**Exercise 3 — Add a reason field:** Modify `HITLState` to include a `reason: str` field. Change `interrupt()` to accept a reason string from the human, and log it in `result` when cancelling.

## What's next

- **5-react-agent-lg** — MemorySaver for persisting multi-turn conversation history
- **30-agentic-rag** — agent with tools; add HITL approval before executing tool calls
- **29-llm-judge-harness** — replace the human with an LLM judge to automate approval decisions